In [8]:
!pip install -q transformers datasets scikit-learn scipy joblib

In [9]:
import contextlib
import io
import random
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from PIL import Image, ImageFilter, ImageEnhance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", use_safetensors=True).to(device)
clip_model.eval()

def get_embedding(output):
    if isinstance(output, torch.Tensor):
        return output
    elif hasattr(output, "image_embeds"):
        return output.image_embeds
    elif hasattr(output, "pooler_output"):
        return output.pooler_output
    else:
        raise ValueError(f"Unexpected output type: {type(output)}")

print(f"Frozen CLIP loaded on {device}.")
clip_params = sum(p.numel() for p in clip_model.parameters())
print(f"CLIP parameters: {clip_params:,} ({clip_params/1e6:.1f}M)")

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Frozen CLIP loaded on cpu.
CLIP parameters: 151,277,313 (151.3M)


In [10]:
def extract_clip_features(images, batch_size=16):
    all_feats = []
    for i in tqdm(range(0, len(images), batch_size)):
        batch = images[i:i+batch_size]
        inputs = clip_processor(images=batch, return_tensors="pt").to(device)
        with torch.no_grad():
            output = clip_model.get_image_features(**inputs)
        emb = get_embedding(output).cpu().numpy()
        all_feats.append(emb)
    return np.concatenate(all_feats, axis=0)

print("CLIP feature extraction function defined.")

CLIP feature extraction function defined.


In [11]:
def jpeg_compress(image, quality):
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=quality, subsampling=2)
    buffer.seek(0)
    with Image.open(buffer) as compressed:
        return compressed.convert("RGB").copy()

def gaussian_blur(image, sigma):
    return image.filter(ImageFilter.GaussianBlur(radius=sigma))

def resize_degrade(image, scale):
    w, h = image.size
    reduced = image.resize((max(1, round(w*scale)), max(1, round(h*scale))), Image.Resampling.BICUBIC)
    return reduced.resize((w, h), Image.Resampling.BICUBIC)

def gaussian_noise(image, sigma):
    arr = np.array(image).astype(np.float32) / 255.0
    noise = np.random.normal(0, sigma, arr.shape)
    noisy = np.clip(arr + noise, 0, 1) * 255
    return Image.fromarray(noisy.astype(np.uint8))

def color_jitter(image, factor):
    image = ImageEnhance.Brightness(image).enhance(1 + factor)
    image = ImageEnhance.Contrast(image).enhance(1 + factor)
    return image

def center_crop(image, pct):
    w, h = image.size
    new_w, new_h = int(w*pct), int(h*pct)
    left, top = (w-new_w)//2, (h-new_h)//2
    return image.crop((left, top, left+new_w, top+new_h)).resize((w, h))

transforms_to_test = {
    "clean": lambda img: img,
    "jpeg_q90": lambda img: jpeg_compress(img, 90),
    "jpeg_q50": lambda img: jpeg_compress(img, 50),
    "jpeg_q30": lambda img: jpeg_compress(img, 30),
    "blur_sigma1": lambda img: gaussian_blur(img, 1.0),
    "blur_sigma2": lambda img: gaussian_blur(img, 2.0),
    "resize_0.5x": lambda img: resize_degrade(img, 0.5),
    "resize_0.25x": lambda img: resize_degrade(img, 0.25),
    "noise_0.05": lambda img: gaussian_noise(img, 0.05),
    "noise_0.10": lambda img: gaussian_noise(img, 0.10),
    "color_jitter": lambda img: color_jitter(img, 0.2),
    "center_crop80": lambda img: center_crop(img, 0.8),
}

print("Transforms defined.")

Transforms defined.


In [12]:
FFT_BINS = 32

def fft_radial_features(image, bins=FFT_BINS):
    array = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    height, width, _ = array.shape
    yy, xx = np.indices((height, width))
    radius = np.sqrt((yy - (height-1)/2.0)**2 + (xx - (width-1)/2.0)**2)
    radius = radius / max(float(radius.max()), 1.0)
    edges = np.linspace(0.0, 1.0, bins + 1)
    features = []
    for channel in range(3):
        magnitude = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(array[:, :, channel]))))
        for i in range(bins):
            if i == bins - 1:
                mask = (radius >= edges[i]) & (radius <= edges[i+1])
            else:
                mask = (radius >= edges[i]) & (radius < edges[i+1])
            features.append(float(magnitude[mask].mean()) if mask.any() else 0.0)
    return np.asarray(features, dtype=np.float32)

print("FFT radial feature function defined.")

FFT radial feature function defined.


In [13]:
def collect_sid_examples(stream, n_per_class, classes=(0, 1, 2), max_size=384):
    counts = {c: 0 for c in classes}
    collected = []
    for example in stream:
        label = example["label"]
        if label in classes and counts[label] < n_per_class:
            img = example["image"].convert("RGB")
            img.thumbnail((max_size, max_size))
            binary_label = 0 if label == 0 else 1
            collected.append((img, binary_label))
            counts[label] += 1
        if all(c >= n_per_class for c in counts.values()):
            break
    return collected

sid_train_stream = load_dataset("saberzl/SID_Set", split="train", streaming=True)
sid_val_stream = load_dataset("saberzl/SID_Set", split="validation", streaming=True)

sid_train_items = collect_sid_examples(sid_train_stream, n_per_class=500)
sid_val_items = collect_sid_examples(sid_val_stream, n_per_class=200)

print("Train items:", len(sid_train_items))
print("Val items:", len(sid_val_items))

Resolving data files:   0%|          | 0/249 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/249 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/249 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/249 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

Train items: 1500
Val items: 600


In [14]:
train_images = [img for img, _ in sid_train_items]
train_labels = np.array([label for _, label in sid_train_items])
val_images = [img for img, _ in sid_val_items]
val_labels = np.array([label for _, label in sid_val_items])

print("Extracting CLIP features (train)...")
train_clip = extract_clip_features(train_images)
print("Extracting CLIP features (val)...")
val_clip = extract_clip_features(val_images)

C_GRID = (0.01, 0.1, 1.0, 10.0)
CLASS_WEIGHT_GRID = (None, "balanced")

def make_probe(C, class_weight):
    return Pipeline([
        ("scale", StandardScaler()),
        ("classifier", LogisticRegression(C=C, class_weight=class_weight, max_iter=5000, random_state=SEED)),
    ])

folds = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED).split(
    train_clip, y=train_labels, groups=np.arange(len(train_labels))
))

best_auc, best_params = -1, None
for C in C_GRID:
    for cw in CLASS_WEIGHT_GRID:
        aucs = []
        for tr_idx, va_idx in folds:
            probe = make_probe(C, cw)
            probe.fit(train_clip[tr_idx], train_labels[tr_idx])
            probs = probe.predict_proba(train_clip[va_idx])[:, 1]
            aucs.append(roc_auc_score(train_labels[va_idx], probs))
        mean_auc = np.mean(aucs)
        if mean_auc > best_auc:
            best_auc, best_params = mean_auc, (C, cw)

print(f"Best CV AUC: {best_auc:.4f} (C={best_params[0]}, class_weight={best_params[1]})")

spatial_model = make_probe(*best_params)
spatial_model.fit(train_clip, train_labels)
print("Spatial (CLIP) model trained with CV-selected hyperparameters.")

Extracting CLIP features (train)...


  0%|          | 0/94 [00:00<?, ?it/s]

Extracting CLIP features (val)...


  0%|          | 0/38 [00:00<?, ?it/s]

Best CV AUC: 0.9313 (C=0.01, class_weight=None)
Spatial (CLIP) model trained with CV-selected hyperparameters.


In [15]:
print("Extracting FFT radial features (train)...")
train_fft = np.stack([fft_radial_features(img) for img in tqdm(train_images)])
print("Extracting FFT radial features (val)...")
val_fft = np.stack([fft_radial_features(img) for img in tqdm(val_images)])

frequency_model = make_probe(1.0, "balanced")
frequency_model.fit(train_fft, train_labels)
print("Frequency (FFT) model trained.")

spatial_val_probs = spatial_model.predict_proba(val_clip)[:, 1]
frequency_val_probs = frequency_model.predict_proba(val_fft)[:, 1]

best_alpha, best_val_auc = 1.0, roc_auc_score(val_labels, spatial_val_probs)
for alpha in np.linspace(0, 1, 21):
    fused = alpha * spatial_val_probs + (1 - alpha) * frequency_val_probs
    auc = roc_auc_score(val_labels, fused)
    if auc > best_val_auc:
        best_val_auc, best_alpha = auc, alpha

print(f"Best fusion alpha: {best_alpha:.2f} (val AUC: {best_val_auc:.4f})")

Extracting FFT radial features (train)...


  0%|          | 0/1500 [00:00<?, ?it/s]

Extracting FFT radial features (val)...


  0%|          | 0/600 [00:00<?, ?it/s]

Frequency (FFT) model trained.
Best fusion alpha: 0.55 (val AUC: 0.9861)


In [16]:
def run_full_robustness_test_clip_fusion(val_items, spatial_model, frequency_model, alpha):
    results = {}
    for name, transform_fn in transforms_to_test.items():
        transformed = [transform_fn(img) for img, _ in val_items]
        labels = np.array([label for _, label in val_items])

        spatial_feats = extract_clip_features(transformed)
        spatial_probs = spatial_model.predict_proba(spatial_feats)[:, 1]

        fft_feats = np.stack([fft_radial_features(img) for img in transformed])
        freq_probs = frequency_model.predict_proba(fft_feats)[:, 1]

        fused_probs = alpha * spatial_probs + (1 - alpha) * freq_probs
        preds = (fused_probs >= 0.5).astype(int)

        acc = accuracy_score(labels, preds)
        auc = roc_auc_score(labels, fused_probs)
        results[name] = {"acc": acc, "auc": auc}
        print(f"{name}: acc={acc:.4f}, auc={auc:.4f}")
    return results

print("Running FULL 12-condition robustness test on CLIP+FFT gated fusion...")
clip_fusion_results = run_full_robustness_test_clip_fusion(sid_val_items, spatial_model, frequency_model, best_alpha)

Running FULL 12-condition robustness test on CLIP+FFT gated fusion...


  0%|          | 0/38 [00:00<?, ?it/s]

clean: acc=0.9383, auc=0.9861


  0%|          | 0/38 [00:00<?, ?it/s]

jpeg_q90: acc=0.9383, auc=0.9802


...

noise_0.10: acc=0.8133, auc=0.9106


  0%|          | 0/38 [00:00<?, ?it/s]

color_jitter: acc=0.9367, auc=0.9847


  0%|          | 0/38 [00:00<?, ?it/s]

center_crop80: acc=0.9683, auc=0.9928


In [17]:
clean_auc = clip_fusion_results["clean"]["auc"]
robust_avg = np.mean([v["auc"] for k, v in clip_fusion_results.items() if k != "clean"])
final_score = 0.5 * clean_auc + 0.5 * robust_avg

print(f"CLIP+FFT (gated fusion) Clean AUC: {clean_auc:.4f}")
print(f"Average robust AUC: {robust_avg:.4f}")
print(f"FINAL COMBINED SCORE: {final_score:.4f}")
print(f"\nCompare against your original CLIP+FFT (concatenation): 0.9506")
print(f"Compare against DINOv2+FFT (gated fusion): 0.9774")

CLIP+FFT (gated fusion) Clean AUC: 0.9861
Average robust AUC: 0.9732
FINAL COMBINED SCORE: 0.9796

Compare against your original CLIP+FFT (concatenation): 0.9506
Compare against DINOv2+FFT (gated fusion): 0.9774


In [19]:
import getpass

token = getpass.getpass("Paste your Kaggle API token: ")
!mkdir -p ~/.kaggle
with open("/root/.kaggle/access_token", "w") as f:
    f.write(token)
!chmod 600 ~/.kaggle/access_token
!pip install -q kaggle

Paste your Kaggle API token: ··········


In [20]:
!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images -p ./data
!unzip -q -o ./data/*.zip -d ./data

import os
print(os.listdir("./data"))
print(os.listdir("./data/test"))

Dataset URL: https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images
License(s): other
100% 105M/105M [00:03<00:00, 32.9MB/s]

['test', 'cifake-real-and-ai-generated-synthetic-images.zip', 'train']
['REAL', 'FAKE']


In [21]:
# Reuse your existing CIFAKE data (already downloaded earlier tonight) if still on disk,
# otherwise re-download quickly via kaggle as before

import os
cifake_test_dir_real = "./data/test/REAL"
cifake_test_dir_fake = "./data/test/FAKE"

# Sample a manageable batch
real_files = [os.path.join(cifake_test_dir_real, f) for f in os.listdir(cifake_test_dir_real)[:50]]
fake_files = [os.path.join(cifake_test_dir_fake, f) for f in os.listdir(cifake_test_dir_fake)[:50]]

cifake_images = [Image.open(p).convert("RGB") for p in real_files] + [Image.open(p).convert("RGB") for p in fake_files]
cifake_labels = np.array([0]*len(real_files) + [1]*len(fake_files))

# Run through your final gated-fusion pipeline
spatial_feats = extract_clip_features(cifake_images)  # or extract_dino_variants, whichever is your final model
spatial_probs = spatial_model.predict_proba(spatial_feats)[:, 1]

fft_feats = np.stack([fft_radial_features(img) for img in cifake_images])
freq_probs = frequency_model.predict_proba(fft_feats)[:, 1]

fused_probs = best_alpha * spatial_probs + (1 - best_alpha) * freq_probs
preds = (fused_probs >= 0.5).astype(int)

from sklearn.metrics import accuracy_score, roc_auc_score
print("CIFAKE (out-of-distribution) accuracy:", accuracy_score(cifake_labels, preds))
print("CIFAKE (out-of-distribution) AUC:", roc_auc_score(cifake_labels, fused_probs))

  0%|          | 0/7 [00:00<?, ?it/s]

CIFAKE (out-of-distribution) accuracy: 0.52
CIFAKE (out-of-distribution) AUC: 0.5112


In [22]:
import joblib
joblib.dump(spatial_model, "spatial_model.joblib")
joblib.dump(frequency_model, "frequency_model.joblib")
print("Models saved.")
print("Fusion alpha used:", best_alpha)

Models saved.
Fusion alpha used: 0.55


In [23]:
%%writefile infer.py

Writing infer.py


In [24]:
%%writefile infer.py
"""
Required deliverable: takes an image directory, outputs JSON with
image_path and pred (confidence score, 0=real, 1=AI-generated) for each image.

Usage:
    python infer.py --input_dir /path/to/images --output predictions.json --fusion_alpha 0.XX
"""

import argparse
import json
import os

import joblib
import numpy as np
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

MODEL_NAME = "openai/clip-vit-base-patch32"
FFT_BINS = 32


def get_embedding(output):
    if isinstance(output, torch.Tensor):
        return output
    elif hasattr(output, "image_embeds"):
        return output.image_embeds
    elif hasattr(output, "pooler_output"):
        return output.pooler_output
    else:
        raise ValueError(f"Unexpected output type: {type(output)}")


def fft_radial_features(image, bins=FFT_BINS):
    array = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    height, width, _ = array.shape
    yy, xx = np.indices((height, width))
    radius = np.sqrt((yy - (height - 1) / 2.0) ** 2 + (xx - (width - 1) / 2.0) ** 2)
    radius = radius / max(float(radius.max()), 1.0)
    edges = np.linspace(0.0, 1.0, bins + 1)
    features = []
    for channel in range(3):
        magnitude = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(array[:, :, channel]))))
        for i in range(bins):
            if i == bins - 1:
                mask = (radius >= edges[i]) & (radius <= edges[i + 1])
            else:
                mask = (radius >= edges[i]) & (radius < edges[i + 1])
            features.append(float(magnitude[mask].mean()) if mask.any() else 0.0)
    return np.asarray(features, dtype=np.float32)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_dir", required=True)
    parser.add_argument("--output", default="predictions.json")
    parser.add_argument("--spatial_model", default="spatial_model.joblib")
    parser.add_argument("--frequency_model", default="frequency_model.joblib")
    parser.add_argument("--fusion_alpha", type=float, default=0.7)
    parser.add_argument("--batch_size", type=int, default=32)
    args = parser.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    print("Loading CLIP...")
    clip_model = CLIPModel.from_pretrained(MODEL_NAME, use_safetensors=True).to(device)
    clip_model.eval()
    clip_processor = CLIPProcessor.from_pretrained(MODEL_NAME)

    print("Loading trained models...")
    spatial_model = joblib.load(args.spatial_model)
    frequency_model = joblib.load(args.frequency_model)

    image_paths = [
        os.path.join(args.input_dir, f)
        for f in sorted(os.listdir(args.input_dir))
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]
    print(f"Found {len(image_paths)} images in {args.input_dir}")

    results = []

    for i in range(0, len(image_paths), args.batch_size):
        batch_paths = image_paths[i:i + args.batch_size]
        images, valid_paths = [], []

        for p in batch_paths:
            try:
                img = Image.open(p).convert("RGB")
                images.append(img)
                valid_paths.append(p)
            except Exception as e:
                print(f"Skipping unreadable image {p}: {e}")
                results.append({"image_path": p, "pred": None})

        if not images:
            continue

        inputs = clip_processor(images=images, return_tensors="pt").to(device)
        with torch.no_grad():
            output = clip_model.get_image_features(**inputs)
        clip_features = get_embedding(output).cpu().numpy()

        fft_features = np.stack([fft_radial_features(img) for img in images])

        spatial_probs = spatial_model.predict_proba(clip_features)[:, 1]
        freq_probs = frequency_model.predict_proba(fft_features)[:, 1]
        fused_probs = args.fusion_alpha * spatial_probs + (1 - args.fusion_alpha) * freq_probs

        for p, prob in zip(valid_paths, fused_probs):
            results.append({"image_path": p, "pred": float(prob)})

    with open(args.output, "w") as f:
        json.dump(results, f, indent=2)

    print(f"Wrote {len(results)} predictions to {args.output}")


if __name__ == "__main__":
    main()

Overwriting infer.py
